# Week 5: Fourier series and the FFT

**What we do.** We compute Fourier coefficients of the square wave, watch its partial sums converge (and the Gibbs phenomenon), implement the discrete Fourier transform (DFT) twice — directly from the definition and with the fast recursive algorithm (FFT) — compare their speed, and look at the cosine transform (DCT).

**What to expect.** The coefficients of the square wave decay like `1/n`; the partial sums overshoot near the jumps by about 9% of the jump, no matter how many terms we take. Both DFT implementations agree with `numpy.fft.fft` up to rounding, but the FFT is dramatically faster for long signals (`N log N` instead of `N²` operations).

**How to read the code.** All functions come from `wavelets_lab.fourier`. The DFT convention is `X[k] = Σ_n x[n] e^{-2πikn/N}`, the same as in NumPy.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from wavelets_lab import fourier

## Fourier coefficients of the square wave

**What we do.** We compute `c_n` for `|n| ≤ 15` of the square wave `sign(sin t)` and plot their absolute values.

**What to expect.** `c_n = 0` for even `n` and `|c_n| = 2/(π|n|)` for odd `n`: a slow `1/n` decay, typical of functions with jumps (smooth functions have much faster decaying coefficients).

**How to read the code.** `fourier.fourier_coefficients(f, n_max)` returns the array `c_{-n_max}, …, c_{n_max}`, so `indices` lists the corresponding `n`. `plt.stem` draws vertical lines at the coefficients; the dashed curve is the theoretical envelope `2/(π|n|)`.

In [ ]:
n_max = 15
coefficients = fourier.fourier_coefficients(fourier.square_wave, n_max)
indices = np.arange(-n_max, n_max + 1)
nonzero = indices != 0
plt.figure(figsize=(9, 3))
plt.stem(indices, np.abs(coefficients))
plt.plot(
    indices[nonzero],
    2 / (np.pi * np.abs(indices[nonzero])),
    "k--",
    label="2 / (pi |n|)",
)
plt.xlabel("index n")
plt.ylabel("|c_n|")
plt.legend()
plt.show()

## Partial sums and the Gibbs phenomenon

**What we do.** We draw the partial sums `S_N(t) = Σ_{|n|≤N} c_n e^{int}` for `N = 5, 21, 101`.

**What to expect.** The partial sums approach the square wave away from the jumps, but next to each jump there is an overshoot of about 1.18 (instead of 1) which becomes narrower, not smaller, as `N` grows: this is the Gibbs phenomenon. The convergence is in `L²`, not uniform.

**How to read the code.** `fourier.fourier_partial_sum(coefficients, t)` evaluates the sum; `.real` drops the imaginary part, which vanishes for a real function. 8192 quadrature points are used for the coefficients, enough for `N = 101`.

In [ ]:
t = np.linspace(-np.pi, np.pi, 2001)
plt.figure(figsize=(10, 4))
plt.plot(t, fourier.square_wave(t), "k", label="square wave")
for n_max in (5, 21, 101):
    coefficients = fourier.fourier_coefficients(fourier.square_wave, n_max, 8192)
    plt.plot(t, fourier.fourier_partial_sum(coefficients, t).real, label=f"N = {n_max}")
plt.xlabel("t")
plt.ylabel("value")
plt.title("Gibbs phenomenon: the overshoot does not vanish")
plt.legend()
plt.show()

## The DFT, its inverse and the FFT

**What we do.** We transform a random signal of length 1024 with the direct DFT and with the radix-2 FFT, and invert the DFT.

**What to expect.** All three differences are of order `1e-11`–`1e-13` (rounding errors): the implementations are correct.

**How to read the code.** `fourier.dft(x)` multiplies by the Fourier matrix; `fourier.fft_radix2(x)` uses the even/odd splitting recursively; `fourier.idft` is the inverse. The cell returns the largest absolute differences.

In [ ]:
x = np.random.default_rng(0).standard_normal(1024)
(
    np.abs(fourier.dft(x) - np.fft.fft(x)).max(),
    np.abs(fourier.fft_radix2(x) - np.fft.fft(x)).max(),
    np.abs(fourier.idft(fourier.dft(x)) - x).max(),
)

## Speed: O(N²) versus O(N log N)

**What we do.** We measure the running time of both algorithms for `N = 256, …, 4096` and plot it on logarithmic axes.

**What to expect.** On a log-log plot a power law `N^p` is a straight line of slope `p`. The direct DFT follows slope 2 (doubling `N` multiplies the time by 4); the FFT grows only slightly faster than linearly. The absolute times depend on the computer.

**How to read the code.** `time.perf_counter()` returns the current time in seconds; the difference of two calls is the elapsed time. Each measurement is repeated 3 times and the minimum is taken to reduce random disturbances.

In [ ]:
sizes = [256, 512, 1024, 2048, 4096]
times_dft, times_fft = [], []
for n in sizes:
    x = np.random.default_rng(n).standard_normal(n)
    for algorithm, times in ((fourier.dft, times_dft), (fourier.fft_radix2, times_fft)):
        best = np.inf
        for _ in range(3):
            start = time.perf_counter()
            algorithm(x)
            best = min(best, time.perf_counter() - start)
        times.append(best)
plt.figure(figsize=(6, 4))
plt.loglog(sizes, times_dft, "o-", label="direct DFT")
plt.loglog(sizes, times_fft, "s-", label="radix-2 FFT")
plt.xlabel("signal length N")
plt.ylabel("time [s]")
plt.legend()
plt.show()

## The cosine transform concentrates energy

**What we do.** We compare the sizes of the DFT and DCT coefficients of the smooth, non-periodic signal `x(t) = t²` on `[0, 1]`.

**What to expect.** The DFT treats the signal as periodic, which creates a jump from `x(1) = 1` back to `x(0) = 0`; its coefficients therefore decay slowly (like `1/k`). The DCT implicitly extends the signal symmetrically, without a jump, so its coefficients decay much faster: a few numbers describe the signal well. This is why JPEG uses the DCT.

**How to read the code.** `fourier.dct2(x)` is the orthonormal DCT-II. The DFT is divided by `sqrt(N)` to put both transforms on the same (orthonormal) scale. Only the first half of the DFT is shown because the second half mirrors it for a real signal. The vertical axis is logarithmic (`yscale("log")`).

In [ ]:
x = np.linspace(0.0, 1.0, 64) ** 2
dft_size = np.abs(np.fft.fft(x))[:32] / np.sqrt(x.size)
dct_size = np.abs(fourier.dct2(x))[:32]
plt.figure(figsize=(9, 4))
plt.semilogy(dft_size, "o", label="|DFT| / sqrt(N)")
plt.semilogy(dct_size, "x", label="|DCT|")
plt.xlabel("coefficient index k")
plt.ylabel("absolute value (log scale)")
plt.legend()
plt.show()

## Reference: scripts and functions used in this notebook

**What we do.** We list every function of the course package used above with its purpose, input, output and a recipe for writing it yourself.

**What to expect.** One module is used: `wavelets_lab.fourier` (`src/wavelets_lab/fourier.py`).

**How to read the code.** The DFT convention is `X[k] = Σ_n x[n] e^{-2πikn/N}` everywhere.

**Library functions called directly in this notebook.**

| Function | What it does | Input → output |
|---|---|---|
| `np.fft.fft(x)` | the FFT of NumPy (reference implementation) | 1D array → complex array |
| `np.random.default_rng(seed).standard_normal(n)` | `n` independent standard normal numbers | seed, length → array |
| `time.perf_counter()` | high-resolution clock | – → seconds (`float`) |
| `plt.stem(x, y)` | vertical lines from 0 to `y` at the positions `x` | arrays → plot |
| `plt.loglog`, `plt.semilogy` | graphs with logarithmic axes (both / vertical only) | arrays → line |

### Module `wavelets_lab.fourier` (file `src/wavelets_lab/fourier.py`)

Fourier series, the discrete Fourier transform (DFT) and Fourier coding of images. The DFT convention is the one of `numpy.fft`:
`X[k] = sum_{n=0}^{N-1} x[n] exp(-2 pi i k n / N)`, inverse `x[n] = (1/N) sum_k X[k] exp(2 pi i k n / N)`, so own implementations can be compared with the library directly. Only NumPy (and `collections.abc.Callable` for type hints) is imported.

#### `fourier.square_wave(t)`

**What it is.** The `2π`-periodic square wave `sign(sin t)`: +1 on `(0, π)`, −1 on `(π, 2π)`, 0 at the jumps. The classic example of a function with jumps.

**Input.** `t` (array-like): points.

**Output.** `float64` array with values in `{-1, 0, 1}`.

**How to write it yourself.** Return `np.sign(np.sin(np.asarray(t, dtype=np.float64)))`.

#### `fourier.fourier_coefficients(f, n_max, n_samples=4096)`

**What it is.** Approximates the Fourier coefficients `c_n = (1/2π) ∫_0^{2π} f(t) e^{-int} dt` of a `2π`-periodic function for `n = -n_max, …, n_max`. The integral is replaced by the rectangle rule on `n_samples` equally spaced points, which is exactly a DFT divided by `n_samples`.

**Input.**
- `f` (callable): vectorized function of one variable (accepts an array, returns an array).
- `n_max` (`int`): largest index.
- `n_samples` (`int`, default 4096): number of quadrature points, must exceed `2 * n_max`.

**Output.** Complex array of length `2 * n_max + 1`; entry `n_max + n` is `c_n` (so the middle entry is `c_0`). Raises `ValueError` if `n_samples <= 2 * n_max`.

**How to write it yourself.**
1. Check `n_samples > 2 * n_max` (otherwise the DFT cannot distinguish the frequencies).
2. Quadrature points: `t = 2 * np.pi * np.arange(n_samples) / n_samples`.
3. Rectangle rule: `(1/N) sum_m f(t_m) e^{-i n t_m}` is the DFT at index `n`, so `spectrum = np.fft.fft(f(t)) / n_samples`.
4. Negative indices `n` are stored at the end of the DFT (index `N + n`); pick them with modular indexing: `indices = np.arange(-n_max, n_max + 1)` and return `spectrum[indices % n_samples]`.

#### `fourier.fourier_partial_sum(coefficients, t)`

**What it is.** Evaluates the partial sum of the Fourier series `S_N f(t) = sum_{|n| <= N} c_n e^{int}`.

**Input.**
- `coefficients` (array-like): coefficients ordered as returned by `fourier_coefficients`.
- `t` (array-like): points.

**Output.** Complex array of the values; take `.real` for a real function.

**How to write it yourself.**
1. `n_max = (len(coefficients) - 1) // 2` and `indices = np.arange(-n_max, n_max + 1)`.
2. Matrix of exponentials, one row per point: `E = np.exp(1j * np.multiply.outer(t, indices))` (shape `(len(t), 2 n_max + 1)`).
3. Return `E @ coefficients`.

#### `fourier.dft(x)`

**What it is.** The DFT computed directly from its definition, i.e. a multiplication by the `N × N` Fourier matrix. Cost `O(N^2)` operations.

**Input.** `x` (array-like): 1D signal of length `N` (real or complex).

**Output.** Complex array `X` of length `N`.

**How to write it yourself.**
1. `signal = np.asarray(x, dtype=np.complex128)`, `n = signal.size`, `k = np.arange(n)`.
2. Fourier matrix: `F = np.exp(-2j * np.pi * np.outer(k, k) / n)`.
3. Return `F @ signal`.

#### `fourier.idft(spectrum)`

**What it is.** The inverse DFT, written with the trick `x = conj(DFT(conj(X))) / N`, so no second matrix is needed.

**Input.** `spectrum` (array-like): DFT coefficients.

**Output.** Complex array: the signal.

**How to write it yourself.**
1. `values = np.asarray(spectrum, dtype=np.complex128)`.
2. Return `np.conj(dft(np.conj(values))) / values.size`.

#### `fourier.fft_radix2(x)`

**What it is.** The fast Fourier transform (Danielson–Lanczos / Cooley–Tukey radix 2), written recursively. Splitting the signal into even and odd samples gives two DFTs of half length `E` and `O`, and
`X[k] = E[k] + w^k O[k]`, `X[k + N/2] = E[k] - w^k O[k]` with `w = e^{-2πi/N}`.
Repeating the split `log2 N` times gives cost `O(N log N)`.

**Input.** `x` (array-like): 1D signal whose length is a power of two.

**Output.** Complex array equal to `np.fft.fft(x)`. Raises `ValueError` if the length is not a power of two.

**How to write it yourself.**
1. Convert to `complex128`; `n = signal.size`.
2. A positive integer is a power of two exactly when `n & (n - 1) == 0`; raise `ValueError` if `n == 0` or this fails.
3. Base case: if `n == 1`, return a copy (the DFT of one sample is the sample).
4. Recursion: `even = fft_radix2(signal[0::2])`, `odd = fft_radix2(signal[1::2])`.
5. Twiddle factors: `twiddle = np.exp(-2j * np.pi * np.arange(n // 2) / n) * odd`.
6. Return `np.concatenate([even + twiddle, even - twiddle])`.

#### `fourier.dct2(x)`

**What it is.** The orthonormal discrete cosine transform of type II (the transform of JPEG), from the definition:
`X[k] = a_k sum_n x[n] cos(π k (2n + 1) / (2N))`, `a_0 = sqrt(1/N)`, `a_k = sqrt(2/N)` for `k > 0`. It equals `scipy.fft.dct(x, norm="ortho")`. For smooth signals it concentrates the energy in few coefficients better than the DFT, because it implicitly extends the signal symmetrically (no jump at the boundary).

**Input.** `x` (array-like): real 1D signal of length `N`.

**Output.** `float64` array of `N` coefficients.

**How to write it yourself.**
1. `signal = np.asarray(x, dtype=np.float64)`, `n = signal.size`, `k = np.arange(n)`.
2. Cosine matrix: `C = np.cos(np.pi * np.outer(k, 2 * k + 1) / (2 * n))` (row `k`, column `n`).
3. Scaling: `a = np.full(n, np.sqrt(2 / n))`, then `a[0] = np.sqrt(1 / n)`.
4. Return `a * (C @ signal)`.